# Chapter 6: Gaussian Distributions

<a href="../lite/lab/index.html?path=ch06_gaussian_distributions.ipynb" target="_blank" style="display:inline-block;padding:8px 18px;background:#1976d2;color:white;border-radius:5px;text-decoration:none;font-weight:bold;font-size:0.95em;">&#9654; Open in JupyterLite — run and edit this notebook</a>

*Runs entirely in your browser — no installation required.*

**How to use:** Edit the parameter values in each cell and re-run it to explore.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from scipy.stats import norm, multivariate_normal

%matplotlib inline
plt.rcParams['figure.figsize'] = (9, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

## 6.1 The 1D Gaussian

A Gaussian distribution over a scalar $x$ is parameterized by two numbers:
- **Mean** $\mu$ — the center, our best estimate
- **Variance** $\sigma^2$ — the spread, our uncertainty

$$p(x) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\!\left(-\frac{(x-\mu)^2}{2\sigma^2}\right)$$

**Try it:** Change `mu` and `sigma` in the cell below and re-run it.

```{admonition} What you will build
:class: tip

- Visualize 1D and 2D Gaussian distributions with interactive parameters
- Draw confidence ellipses that show robot position uncertainty
- Transform a Gaussian through a linear function and see how the ellipse stretches

**Real world application:** Gaussians are the default model for sensor noise and state uncertainty. After this chapter, you will be able to read and draw the uncertainty ellipses that appear in every SLAM paper.
```

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
mu    = 0.0   # mean: center of the distribution   (try -3, 0, 2)
sigma = 1.5   # std deviation: spread              (try 0.5, 1, 3)
# ────────────────────────────────────────────────────────────────────────────

x = np.linspace(-10, 10, 500)
y = norm.pdf(x, mu, sigma)

fig, ax = plt.subplots()
ax.plot(x, y, 'steelblue', linewidth=2.5)
ax.axvline(mu, color='tomato', linestyle='--', label=f'mean μ = {mu:.1f}')
ax.axvspan(mu - sigma,   mu + sigma,   alpha=0.15, color='steelblue', label=f'±1σ  (σ = {sigma:.1f})')
ax.axvspan(mu - 2*sigma, mu + 2*sigma, alpha=0.07, color='steelblue', label='±2σ')
ax.set_xlim(-10, 10)
ax.set_ylim(0, None)
ax.set_xlabel('x'); ax.set_ylabel('p(x)')
ax.set_title('1D Gaussian')
ax.legend()
plt.tight_layout()
plt.show()

**Key observations:**
- 68% of probability mass lies within ±1σ of the mean
- 95% lies within ±2σ
- 99.7% lies within ±3σ

When a robot says "I am at position 3.2m with uncertainty σ = 0.5m," it means there is a 68% chance its true position is in [2.7m, 3.7m].

## 6.2 The Multivariate Gaussian

A robot's state is never just one number. Even a planar robot has pose $(x, y, \theta)$ — three correlated uncertain quantities. The multivariate Gaussian extends to $n$ dimensions:

$$p(\mathbf{x}) = \frac{1}{(2\pi)^{n/2}|\boldsymbol{\Sigma}|^{1/2}} \exp\!\left(-\frac{1}{2}(\mathbf{x}-\boldsymbol{\mu})^\top \boldsymbol{\Sigma}^{-1} (\mathbf{x}-\boldsymbol{\mu})\right)$$

where $\boldsymbol{\mu} \in \mathbb{R}^n$ is the mean vector and $\boldsymbol{\Sigma} \in \mathbb{R}^{n \times n}$ is the covariance matrix.

## 6.3 Mean, Covariance, and Correlation

The covariance matrix encodes both individual uncertainties and their correlation:

$$\boldsymbol{\Sigma} = \begin{bmatrix} \sigma_x^2 & \rho\sigma_x\sigma_y \\ \rho\sigma_x\sigma_y & \sigma_y^2 \end{bmatrix}$$

The correlation $\rho \in [-1,1]$ controls the tilt of the uncertainty ellipse. At $\rho=0$ the ellipse is axis-aligned; at $\rho=\pm1$ it collapses to a line.

**Try it:** Change the parameters below.

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
sigma_x = 2.0   # uncertainty in x  (try 0.5 → 4.0)
sigma_y = 1.0   # uncertainty in y  (try 0.5 → 4.0)
rho     = 0.0   # correlation       (try -0.9, 0.0, 0.7, 0.95)
# ────────────────────────────────────────────────────────────────────────────

rho = np.clip(rho, -0.99, 0.99)
cov = np.array([[sigma_x**2, rho*sigma_x*sigma_y],
                [rho*sigma_x*sigma_y, sigma_y**2]])

x = np.linspace(-6, 6, 200)
y = np.linspace(-6, 6, 200)
X, Y = np.meshgrid(x, y)
pos = np.stack([X, Y], axis=-1)
Z = multivariate_normal(mean=[0, 0], cov=cov).pdf(pos)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ax = axes[0]
ax.contourf(X, Y, Z, levels=20, cmap='Blues')
ax.contour(X, Y, Z, levels=5, colors='steelblue', alpha=0.6, linewidths=1)

evals, evecs = np.linalg.eigh(cov)
angle = np.degrees(np.arctan2(evecs[1,1], evecs[0,1]))
for ns, color, lbl in [(1,'tomato','1σ'),(2,'orange','2σ'),(3,'gold','3σ')]:
    ell = patches.Ellipse((0,0), 2*ns*np.sqrt(evals[1]), 2*ns*np.sqrt(evals[0]),
                          angle=angle, fill=False, edgecolor=color, linewidth=2, label=lbl)
    ax.add_patch(ell)
ax.set_xlim(-6,6); ax.set_ylim(-6,6); ax.set_aspect('equal')
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title(f'2D Gaussian  (ρ = {rho:.2f})')
ax.legend(loc='upper right', fontsize=8)

ax2 = axes[1]; ax2.axis('off')
ax2.text(0.1, 0.5,
    f'Σ = [ {cov[0,0]:.2f}   {cov[0,1]:.2f} ]\n'
    f'    [ {cov[1,0]:.2f}   {cov[1,1]:.2f} ]\n\n'
    f'Eigenvalues: {evals[0]:.2f}, {evals[1]:.2f}\n(half-axes of ellipse = √λ)',
    transform=ax2.transAxes, fontsize=13, verticalalignment='center',
    fontfamily='monospace', bbox=dict(boxstyle='round', facecolor='#f0f4f8', alpha=0.8))
plt.tight_layout(); plt.show()

**Things to try:**
1. Set `sigma_x = sigma_y` and vary `rho` — the ellipse rotates.
2. Set `rho = 0` — the ellipse stays axis-aligned (independent variables).
3. Set `rho = 0.9` — models "if the robot is further right, it is probably also further forward."

## 6.4 Confidence Ellipses

| Ellipse | Probability contained |
|---------|----------------------|
| 1σ | 39.3% |
| 2σ | 86.5% |
| 3σ | 98.9% |

In SLAM visualizations you will see 2σ ellipses drawn around the robot pose and landmark estimates. A large ellipse means high uncertainty.

## 6.5 Linear Transformation of a Gaussian

If $\mathbf{x} \sim \mathcal{N}(\boldsymbol{\mu}, \boldsymbol{\Sigma})$ and we apply $\mathbf{y} = A\mathbf{x} + \mathbf{b}$:

$$\mathbf{y} \sim \mathcal{N}(A\boldsymbol{\mu} + \mathbf{b},\ A\boldsymbol{\Sigma}A^\top)$$

The result is still Gaussian. This closed-form propagation is what makes the Kalman filter exact.

**Try it:** Change rotation and scale below.

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
angle_deg = 45.0   # rotation in degrees  (try 0, 30, 90, 135)
scale_x   = 1.0    # scale along x        (try 0.5, 1.5, 2.0)
scale_y   = 1.0    # scale along y        (try 0.5, 1.5, 2.0)
# ────────────────────────────────────────────────────────────────────────────

Sigma = np.array([[3.0, 0.0], [0.0, 0.5]])   # original: elongated along x
theta = np.radians(angle_deg)
R = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
A = R @ np.diag([scale_x, scale_y])
Sigma_new = A @ Sigma @ A.T

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (S, title) in zip(axes, [
    (Sigma,     'Original  $\\mathbf{x} \\sim \\mathcal{N}(0, \\Sigma)$'),
    (Sigma_new, f'Transformed  $A\\mathbf{{x}}$  ({angle_deg}°, {scale_x:.1f}×{scale_y:.1f})')
]):
    xg = np.linspace(-6,6,200); yg = np.linspace(-6,6,200)
    Xg,Yg = np.meshgrid(xg,yg)
    Z = multivariate_normal(mean=[0,0], cov=S).pdf(np.stack([Xg,Yg],axis=-1))
    ax.contourf(Xg, Yg, Z, levels=15, cmap='Blues')
    ev, evec = np.linalg.eigh(S)
    ang = np.degrees(np.arctan2(evec[1,1], evec[0,1]))
    for ns,c in [(1,'tomato'),(2,'orange')]:
        ax.add_patch(patches.Ellipse((0,0), 2*ns*np.sqrt(ev[1]), 2*ns*np.sqrt(ev[0]),
                                    angle=ang, fill=False, edgecolor=c, linewidth=2))
    ax.set_xlim(-6,6); ax.set_ylim(-6,6); ax.set_aspect('equal')
    ax.set_title(title); ax.set_xlabel('x'); ax.set_ylabel('y')
plt.tight_layout(); plt.show()

## Exercises

**Exercise 6.1:** Build a 2D Gaussian with mean $[2, 3]$, $\sigma_x=1$, $\sigma_y=2$, $\rho=0.7$. Construct $\boldsymbol{\Sigma}$ and print it.

In [ ]:
mu = np.array([2, 3])
sigma_x, sigma_y, rho = 1.0, 2.0, 0.7

Sigma = ...  # fill this in

print('Covariance matrix:')
print(Sigma)

**Exercise 6.2:** Sample 500 points from your Gaussian and scatter-plot them. Do they fill the 2σ ellipse?

In [ ]:
# Hint: np.random.multivariate_normal(mean, cov, n_samples)
# Your code here